# Chapter 3 — Instruction-Level Parallelism

This notebook compares one, two, and four independent arithmetic chains inside each thread. Every run uses the same 512 values, 8,192 updates per value, and one block of 32 threads (one warp). The single-warp launch limits thread-level parallelism so the effect of independent chains is easier to study. This is a teaching experiment, not a production launch recommendation.


In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sun Nov  8 15:31:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile code/cuda_check.cuh
#ifndef BOOK_CUDA_CHECK_CUH
#define BOOK_CUDA_CHECK_CUH

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define CUDA_CHECK(call) do {                                      \
    const cudaError_t status_ = (call);                            \
    if (status_ != cudaSuccess) {                                  \
        fprintf(stderr, "%s:%d: %s failed: %s\n",                \
                __FILE__, __LINE__, #call,                         \
                cudaGetErrorString(status_));                      \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while (0)

#endif


Writing code/cuda_check.cuh


## The controlled comparison

Each thread processes 16 values in groups of ILP values. With ILP 1, it updates one value at a time; with ILP 2 or 4, it interleaves updates to that many independent values. Each value receives 8,192 multiply-add updates in every run. Only the number of independent chains available within a thread changes. The template parameter selects a compile-time kernel variant; Chapter 3 explains what that means.


## What this experiment isolates

The 32-thread launch uses one warp, and N = 512 is exactly 32 threads × 16 values per thread. The short vector keeps the test to one warp; it does not mean ILP works only for small arrays. The repeated update loop uses values already loaded into registers, so this test isolates arithmetic dependency latency rather than overlap with global-memory transfers. Real kernels with many ready warps may gain less from extra independent chains. Use profiling and timing on the target workload to decide whether ILP helps.


In [3]:
%%writefile code/ILP_Part_I.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include "cuda_check.cuh"

constexpr int THREADS = 32;
constexpr int ELEMENTS_PER_THREAD = 16;
constexpr int N = THREADS * ELEMENTS_PER_THREAD;
constexpr int N_ITERATIONS = 8192;
constexpr int REPEATS = 100;

template<int ILP>
__global__ void fma_chains(const float *a, const float *b, const float *c,
                           float *out)
{
    const int tid = threadIdx.x;

    // Each thread processes 16 values in groups of ILP independent chains.
    for (int group = 0; group < ELEMENTS_PER_THREAD / ILP; ++group) {
        float accum[ILP];
        float multiplier[ILP];
        float increment[ILP];

        #pragma unroll
        for (int j = 0; j < ILP; ++j) {
            const int i = (group * ILP + j) * blockDim.x + tid;
            accum[j] = a[i];
            multiplier[j] = b[i];
            increment[j] = c[i];
        }

        for (int k = 0; k < N_ITERATIONS; ++k) {
            #pragma unroll
            for (int j = 0; j < ILP; ++j) {
                accum[j] = accum[j] * multiplier[j] + increment[j];
            }
        }

        #pragma unroll
        for (int j = 0; j < ILP; ++j) {
            const int i = (group * ILP + j) * blockDim.x + tid;
            out[i] = accum[j];
        }
    }
}

template<int ILP>
void run_configuration(const float *d_a, const float *d_b, const float *d_c,
                       float *d_out, float *h_out, size_t bytes,
                       cudaEvent_t start, cudaEvent_t stop)
{
    static_assert(ELEMENTS_PER_THREAD % ILP == 0,
                  "ILP must divide ELEMENTS_PER_THREAD");

    // Warm-up: the launch and synchronization are outside the timed batch.
    fma_chains<ILP><<<1, THREADS>>>(d_a, d_b, d_c, d_out);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaEventRecord(start, 0));
    for (int k = 0; k < REPEATS; ++k) {
        fma_chains<ILP><<<1, THREADS>>>(d_a, d_b, d_c, d_out);
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop, 0));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float total_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&total_ms, start, stop));
    if (!(total_ms > 0.0f)) {
        fprintf(stderr, "Invalid timer result.\n");
        exit(EXIT_FAILURE);
    }

    CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));
    const float expected = 0.5f + N_ITERATIONS * 0.25f;
    for (int i = 0; i < N; ++i) {
        if (h_out[i] != expected) {
            fprintf(stderr, "Validation failed for ILP=%d at index %d.\n", ILP, i);
            exit(EXIT_FAILURE);
        }
    }

    printf("ILP = %d, one block, one warp, time = %.6f ms/launch, "
           "validation = PASS\n",
           ILP, total_ms / REPEATS);
}

int main()
{
    const size_t bytes = (size_t)N * sizeof(float);
    float *h_a = (float *)malloc(bytes);
    float *h_b = (float *)malloc(bytes);
    float *h_c = (float *)malloc(bytes);
    float *h_out = (float *)malloc(bytes);
    if (h_a == NULL || h_b == NULL || h_c == NULL || h_out == NULL) {
        fprintf(stderr, "Could not allocate host arrays.\n");
        free(h_a); free(h_b); free(h_c); free(h_out);
        return EXIT_FAILURE;
    }

    for (int i = 0; i < N; ++i) {
        h_a[i] = 0.5f;
        h_b[i] = 1.0f;
        h_c[i] = 0.25f;
    }

    float *d_a = NULL, *d_b = NULL, *d_c = NULL, *d_out = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_out, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_c, h_c, bytes, cudaMemcpyHostToDevice));

    cudaEvent_t start = nullptr, stop = nullptr;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    printf("N = %d, updates per element = %d, timed launches = %d\n",
           N, N_ITERATIONS, REPEATS);
    run_configuration<1>(d_a, d_b, d_c, d_out, h_out, bytes, start, stop);
    run_configuration<2>(d_a, d_b, d_c, d_out, h_out, bytes, start, stop);
    run_configuration<4>(d_a, d_b, d_c, d_out, h_out, bytes, start, stop);

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c)); CUDA_CHECK(cudaFree(d_out));
    free(h_a); free(h_b); free(h_c); free(h_out);
    return EXIT_SUCCESS;
}


Writing code/ILP_Part_I.cu


## Compile

The error-checking header is included by the single CUDA source file.

In [4]:
!nvcc -O2 -std=c++14 -arch=native -Icode code/ILP_Part_I.cu -o ilp_part_i


## Run

Each variant gets one untimed warm-up launch. CUDA events then time a batch of 100 launches. After the batch, the program checks all 512 output values. The reported time is the batch time divided by 100.


In [5]:
!./ilp_part_i


N = 512, updates per element = 8192, timed launches = 100
ILP = 1, one block, one warp, time = 0.627922 ms/launch, validation = PASS
ILP = 2, one block, one warp, time = 0.319269 ms/launch, validation = PASS
ILP = 4, one block, one warp, time = 0.317350 ms/launch, validation = PASS
